In [ ]:
import sys, os
sys.path.append('./src/')

from VAE_trainers import EpochPyroTrainer, AdversarialEpochPyroTrainer, ThresholdPyroTrainer, AdversarialThresholdPyroTrainer
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm, trange
from umap import UMAP
from CNN_variants import CNNDecoder, CNNEncoder, GaussianCNNEncoder, CNNVAE, CNNCVAE, CNNCSVAENA, CNNCSVAE, CNNHCSVAENA, CNNHCSVAE, CNNSDIVA, CNNCCVAE, CNNDLVAE
import pyro.distributions as dist


import torch, pyro
import numpy as np
import matplotlib.pyplot as plt
import copy, cv2
import pyro.optim as opt
from torchvision.datasets import MNIST
from torchvision import transforms
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

# Define the hex codes for the colormap
hex_colors = ["#F23E2E", "#5888A6"]

# Create the colormap
cmap = LinearSegmentedColormap.from_list("cmap", hex_colors)

demo_epochs = 200
noise = 0

## Data

In [ ]:
train_set, test_set = MNIST('./data', train=True, download=True, transform=transforms.ToTensor()), MNIST('./data', train=False, download=True, transform=transforms.ToTensor()) 

In [ ]:
import torch.utils.data as utils

def colorize_and_makedata(mnist_set, batch_size=128, test=False):
    ims, labels, unnoised_labels, numbers, = [], [], [], []
    
    for im_base, label in tqdm(mnist_set, total=len(mnist_set)):
        num = label
        im_base = cv2.cvtColor(np.einsum('ijk -> jki', im_base.numpy()), cv2.COLOR_GRAY2BGR)

        labs, css = (0,1), ([1], [0])

        for label, cs in zip(labs, css):
            im = im_base.copy()
            mask = np.any(im != 0, axis=-1)
            masked_portion = im[mask]
            masked_portion[:,cs] = 0.
            im[mask] = masked_portion
        
        
            
            ims.append(np.einsum('jki -> ijk', im))

            if np.random.uniform() < noise:
                label_exp = abs(label-1) # swap

            else:
                label_exp = label
            
            labels.append(label_exp)
            unnoised_labels.append(label)
            numbers.append(num)

    ims, labels, unnoised_labels, numbers = torch.FloatTensor(np.array(ims)), torch.FloatTensor(np.array(labels)).reshape(-1,1), torch.FloatTensor(np.array(unnoised_labels)).reshape(-1,1), torch.FloatTensor(np.array(numbers)).reshape(-1,1)
    extras = torch.hstack((numbers, unnoised_labels))
    
    dataset = utils.TensorDataset(ims[:len(ims)//4], labels[:len(ims)//4], extras[:len(ims)//4])  # Cut sizes even more for faster run in demo
    
    if not test:
        loader = torch.utils.data.DataLoader(dataset, shuffle=True, batch_size=batch_size)

    else:
        loader = torch.utils.data.DataLoader(dataset, shuffle=False, batch_size=batch_size)

    
    return dataset, loader


(train_set, train_loader), (test_set, test_loader) = colorize_and_makedata(train_set), colorize_and_makedata(test_set, test=True)

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

to_draw = list(range(10))

i,j = 0,0

while i < 10:
    im, label, _ = test_set[j]
    true_label = _[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0 and _[0] == to_draw[0]:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        ax[i//10][i%10].axis('off')
        to_draw.pop(0)
        i += 1

    j += 1


to_draw = list(range(10))


while i < 20:
    im, label, _ = test_set[j]
    true_label = _[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1 and _[0] == to_draw[0]:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        ax[i//10][i%10].axis('off')
        to_draw.pop(0)
        i += 1

    j += 1


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

to_draw = list(range(10))


while i < 10:
    im, label, extras = test_set[j]
    true_label = extras[1]
    
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    im_common = im.copy()
    im_common[..., 1] = im_common[..., 1] / 2
    im_common[..., 0] = im_common[..., 1] 


    if true_label > 0 and extras[0] == to_draw[0]:
        ax[i//10][i%10].imshow(im_common.reshape(28,28,3))
        ax[i//10][i%10].axis('off')
        to_draw.pop(0)
        i += 1

    j += 1


to_draw = list(range(10))


while i < 20:
    im, label, extras = test_set[j]
    true_label = extras[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    im_common = im.copy()
    im_common[..., 0] = im_common[..., 0] / 2
    im_common[..., 1] = im_common[..., 0] 



    


    if true_label < 1 and extras[0] == to_draw[0]:
        ax[i//10][i%10].imshow(im_common.reshape(28,28,3))
        ax[i//10][i%10].axis('off')
        to_draw.pop(0)
        i += 1

    j += 1


## CSVAE - No Adv.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvaena = CNNCSVAENA((28,28), 3, [1], latent_dim=20, w_dim=2, channels=[32,64,128], repeats=[2,1,1], cnn_arch='conv+pool', recon_weight=1, z_kl_weight=1e-4, w_kl_weight=1)
csvaena_trainer = EpochPyroTrainer(demo_epochs, csvaena, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
csvaena_trainer.train()

In [ ]:
preds = csvaena_trainer.predictive(*csvaena_trainer._send_args_to_device(csvaena_trainer.test_loader.dataset[:1000], csvaena_trainer.device))
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
recon_test = torch.stack([csvaena_trainer.best_model.decoder(torch.concatenate((z_s, torch.stack([torch.normal(0, 0.1, (w_s.shape[-1],)) if np.random.choice([0,1], 1) else torch.normal(3, 1, (w_s.shape[-1],)) for elem in test_set[:1000][1]])), dim=-1).cuda()).detach().cpu() for i in range(1000)]).mean(dim=0)
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
reducer = UMAP()
umap_zs = reducer.fit_transform(z_s)

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

## CSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)


csvae = CNNCSVAE((28,28), 3, [1], latent_dim=20, w_dim=2, channels=[32,64,128], repeats=[2,1,1], cnn_arch='conv+pool', recon_weight=1, z_kl_weight=1e-4, w_kl_weight=1, adversarial_weight=1)
csvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, csvae, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
csvae_trainer.train()

In [ ]:
preds = csvae_trainer.predictive(*csvae_trainer._send_args_to_device(csvae_trainer.test_loader.dataset[:1000], csvae_trainer.device))
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
recon_test = torch.stack([csvae_trainer.best_model.decoder(torch.concatenate((z_s, torch.stack([torch.normal(0, 0.1, (w_s.shape[-1],)) if np.random.choice([0,1], 1) else torch.normal(3, 1, (w_s.shape[-1],)) for elem in test_set[:1000][1]])), dim=-1).cuda()).detach().cpu() for i in range(1000)]).mean(dim=0)
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
reducer = UMAP()
umap_zs = reducer.fit_transform(z_s)

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

## HCSVAE - No Adv.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvaena = CNNHCSVAENA((28,28), 3, [1], latent_dim=20, w_dim=2, channels=[32,64,128], repeats=[2,1,1], cnn_arch='conv+pool', recon_weight=1e3, z_kl_weight=1e-4, w_kl_weight=1)
hcsvaena_trainer = EpochPyroTrainer(demo_epochs, hcsvaena, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
hcsvaena_trainer.train()

In [ ]:
preds = hcsvaena_trainer.predictive(*hcsvaena_trainer._send_args_to_device(hcsvaena_trainer.test_loader.dataset[:1000], hcsvaena_trainer.device))
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
recon_test = torch.stack([hcsvaena_trainer.best_model.decoder(hcsvaena_trainer.best_model.decoder_rho(torch.concatenate((z_s, torch.stack([torch.normal(0, 0.1, (w_s.shape[-1],)) if np.random.choice([0,1], 1) else torch.normal(3, 1, (w_s.shape[-1],)) for elem in test_set[:1000][1]])), dim=-1).cuda())[0]).detach().cpu() for i in range(1000)]).mean(dim=0)
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
reducer = UMAP()
umap_zs = reducer.fit_transform(z_s)

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

## HCSVAE 

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvae = CNNHCSVAE((28,28), 3, [1], latent_dim=20, w_dim=2, channels=[32,64,128], repeats=[2,1,1], cnn_arch='conv+pool', recon_weight=1e4, z_kl_weight=1e-4, w_kl_weight=1, adversarial_weight=1)
hcsvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, hcsvae, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
hcsvae_trainer.train()

In [ ]:
preds = hcsvae_trainer.predictive(*hcsvae_trainer._send_args_to_device(hcsvae_trainer.test_loader.dataset[:1000], hcsvae_trainer.device))
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
rho_s = preds['rho'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
recon_test = torch.stack([hcsvae_trainer.best_model.decoder(hcsvae_trainer.best_model.decoder_rho(torch.concatenate((z_s, torch.stack([torch.normal(0, 0.1, (w_s.shape[-1],)) if np.random.choice([0,1], 1) else torch.normal(3, 1, (w_s.shape[-1],)) for elem in test_set[:1000][1]])), dim=-1).cuda())[0]).detach().cpu() for i in range(1000)]).mean(dim=0)
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
reducer = UMAP()
umap_zs = reducer.fit_transform(z_s)

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

## DIVA

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

diva = CNNSDIVA((28,28), 3, [1], latent_dim=20, w_dim=2, channels=[32,64,128], repeats=[2,1,1], cnn_arch='conv+pool', recon_weight=1, kl_weight=1e-4)
diva_trainer = EpochPyroTrainer(demo_epochs, diva, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
diva_trainer.train()

In [ ]:
preds = diva_trainer.predictive(*diva_trainer._send_args_to_device(diva_trainer.test_loader.dataset[:1000], diva_trainer.device))
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
recon_test = torch.stack([diva_trainer.best_model.decoder(torch.concatenate((z_s, dist.Normal(*diva_trainer.best_model.prior_w(torch.FloatTensor(np.random.choice([0,1], (z_s.shape[0], 1))).cuda())).sample().detach().cpu()), dim=-1).cuda()).detach().cpu() for i in trange(1000)]).mean(dim=0)
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
reducer = UMAP()
umap_zs = reducer.fit_transform(z_s)

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

## CCVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

ccvae = CNNCCVAE((28,28), 3, [1], latent_dim=20, w_dim=2, channels=[32,64,128], repeats=[2,1,1], cnn_arch='conv+pool', recon_weight=1, kl_weight=1e-4)
ccvae_trainer = EpochPyroTrainer(demo_epochs, ccvae, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
ccvae_trainer.train()

In [ ]:
preds = ccvae_trainer.predictive(*ccvae_trainer._send_args_to_device(ccvae_trainer.test_loader.dataset[:1000], ccvae_trainer.device))
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
recon_test = torch.stack([ccvae_trainer.best_model.decoder(torch.concatenate((z_s, dist.Normal(*ccvae_trainer.best_model.prior_w(torch.FloatTensor(np.random.choice([0,1], (z_s.shape[0], 1))).cuda())).sample().detach().cpu()), dim=-1).cuda()).detach().cpu() for i in trange(1000)]).mean(dim=0)
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recon_test[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0

while i < 10:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1

while i < 20:
    im, label, num = recons[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


In [ ]:
reducer = UMAP()
umap_zs = reducer.fit_transform(z_s)

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

## Ours

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

dlvae = CNNDLVAE((28,28), 3, [1], latent_dim=20, w_dim=2, channels=[32,64,128], repeats=[2,1,1], cnn_arch='conv+pool', recon_weight=5e-1, recon_weight_z=5e-1, w_kl_weight=1e-4, z_kl_weight=1e-4, adversarial_weight=1e-1, learnable_prior=True)
dlvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, dlvae, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
dlvae_trainer.train()

In [ ]:
preds = dlvae_trainer.predictive(*dlvae_trainer._send_args_to_device(dlvae_trainer.test_loader.dataset[:1000], dlvae_trainer.device))
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons_z = preds['rec_z'][0, 0].cpu()
recons_w = preds['rec_w'][0, 0].cpu()

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0


while i < 10:
    im, label, num = recons_z[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        i += 1

    j += 1


while i < 20:
    im, label, num = recons_z[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')

        i += 1

    j += 1


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

i,j = 0,0
to_draw = list(range(10))


while i < 10:
    im, label, num = recons_w[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    if true_label > 0 and num[0]==to_draw[0]:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
        #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        to_draw.pop(0)
        i += 1

    j += 1

to_draw = list(range(10))

while i < 20:
    im, label, num = recons_w[j], test_set[j][1], test_set[j][2]
    true_label = num[1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    if true_label < 1 and num[0]==to_draw[0]:
        ax[i//10][i%10].imshow(im.reshape(28,28,3))
       #ax[i//10][i%10].set_title(f'{int(num)}')
        ax[i//10][i%10].axis('off')
        to_draw.pop(0)
        i += 1

    j += 1

In [ ]:
reducer = UMAP()
umap_zs = reducer.fit_transform(z_s)

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(umap_zs[:, 0], umap_zs[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:1000][2][:,0].numpy(), cmap='tab10', alpha=0.6)
plt.axis('off')
plt.show()